# Relevance Training on Google Colab
Use a free Colab T4 to train the relevance model with Drive-backed checkpoints.


## Flow
- mount Drive
- clone repo
- install relevance training dependencies
- optionally build `v9` or `v10` datasets
- generate a Colab config
- train relevance
- zip/download the final checkpoint


In [1]:
USE_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/fact_checking_system_colab'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os
import shutil

REPO_URL = 'https://github.com/injetiharsha/fact_checking_system.git'
BRANCH = 'feat/reduce-heuristics-phased'
REPO_DIR = '/content/fact_checking_system'

os.chdir('/content')
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())


Cloning into '/content/fact_checking_system'...
remote: Enumerating objects: 825, done.
remote: Counting objects: 100% (825/825), done.
remote: Compressing objects: 100% (454/454), done.
remote: Total 825 (delta 429), reused 736 (delta 342), pack-reused 0 (from 0)
Receiving objects: 100% (825/825), 793.47 KiB | 8.01 MiB/s, done.
Resolving deltas: 100% (429/429), done.
cwd: /content/fact_checking_system


In [3]:
import os, torch
os.environ['PYTHONWARNINGS'] = 'ignore'
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


CUDA available: True
CUDA device: Tesla T4


In [4]:
!pip install -q --upgrade pip
!pip uninstall -y peft bitsandbytes sentence-transformers > /dev/null 2>&1 || true
!pip install -q transformers==4.38.2 datasets==2.17.1 accelerate==0.27.2 scikit-learn==1.4.2 sentencepiece==0.2.0 PyYAML==6.0.2 tqdm==4.66.2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.0 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires tqdm>=4.67, but you have tqdm 4.66.2 which is incompatible.
cuml-cu12 26.2.0 requires scikit-learn>=1.5, but you have scikit-learn 1.4.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2023.10.0 which is incompatible.
hdbscan 0.8.41 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
umap-learn 0.5.11 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
tobler 0.13.0 requires tqdm>=4.67, but you have tqdm 4.66.2 which is incompatible.


In [5]:
# Optional: rebuild v9 locally in Colab
!python training/common/build_relevance_v9_residual.py


python3: can't open file '/content/fact_checking_system/training/common/build_relevance_v9_residual.py': [Errno 2] No such file or directory


In [6]:
# If you have AVeriTeC files available in the repo, rebuild v10 too.
# !python training/common/build_relevance_v10_averitec.py --averitec-file data/public/averitec/train.json --averitec-file data/public/averitec/dev.json


In [7]:
BASE_CONFIG = 'training/configs/relevance_v9.yaml'
# Example for public-data run:
# BASE_CONFIG = 'training/configs/relevance_v10_averitec.yaml'
!python training/common/generate_colab_relevance_config.py --base-config {BASE_CONFIG} --drive-dir {DRIVE_DIR}


python3: can't open file '/content/fact_checking_system/training/common/generate_colab_relevance_config.py': [Errno 2] No such file or directory


In [8]:
COLAB_CONFIG = BASE_CONFIG.replace('.yaml', '_colab.yaml')
!python training/relevance/train.py --config {COLAB_CONFIG}


2026-03-21 16:27:51.697281: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774110471.735807    6837 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774110471.747525    6837 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774110471.776124    6837 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774110471.776157    6837 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774110471.776165    6837 computation_placer.cc:177] computation placer alr

In [9]:
import os
import shutil
from pathlib import Path

run_name = Path(COLAB_CONFIG).stem.replace('_colab', '')
checkpoint_dir = Path(DRIVE_DIR) / 'checkpoints' / 'relevance' / run_name
archive_path = f'/content/{run_name}.zip'
if checkpoint_dir.exists():
    shutil.make_archive(f'/content/{run_name}', 'zip', checkpoint_dir)
    print('Created:', archive_path)
else:
    print('Checkpoint not found:', checkpoint_dir)


Checkpoint not found: /content/drive/MyDrive/fact_checking_system_colab/checkpoints/relevance/relevance_v9
